# Data Preprocessing cho VGG16
Mục tiêu:
- Khai báo các library
- Thiết lập pipeline biến đổi ảnh, từ 28x28 RW của Dataset qua 224x224 RGB của VGG16
- Tải bộ dữ liệu và chia theo quy tắc 60-20-20 và leak-free
- Lưu indices về file để đảm bảo các bước sau xài chung 1 tập dữ liệu thống nhất

In [1]:
import torch
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import os

# Tạo folder để lưu trữ
os.makedirs("data_splits", exist_ok=True)
print("Đã tải xong thư viện và tạo thư mục lưu trữ.")

Đã tải xong thư viện và tạo thư mục lưu trữ.


## Khai báo các phép biến đổi ảnh (Transforms)
VGG16 yêu cầu đầu vào rất khắt khe, phải áp dụng đúng chỉ số Normalize của tập ImageNet thì mô hình Transfer Learning mới phát huy tác dụng. Ta sẽ chia làm 2 bộ transform:
- **Train Transform:** Có thêm các phép xoay lật nhẹ (Augmentation) để giảm Overfitting.
- **Eval Transform:** Dùng cho Validation và Test, tuyệt đối không làm méo ảnh để đánh giá khách quan nhất.

In [4]:
# Thông số chuẩn hóa (Normalize) bắt buộc của cấu trúc VGG16 (từ ImageNet)
# Normalize ảnh dựa theo mean (average value) và std (độ lệch chuẩn) của ImageNet
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Transform cho tập Train (Có Augmentation)
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Transform cho tập Validation & Test (Không Augmentation)
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

## Tải dữ liệu và Chia tập (Train/Val/Test Split)
Bộ FashionMNIST mặc định cấp sẵn 60,000 ảnh Train và 10,000 ảnh Test.

Để tuân thủ Leak-Free, ta sẽ "khóa" 10,000 ảnh Test lại. Sau đó, trích 20% từ 60,000 ảnh Train gốc để tạo thành tập Validation. Việc chia này phải dùng `stratify` để đảm bảo tỷ lệ các loại quần áo đồng đều.

In [5]:
# Tải toàn bộ tập Train (60.000 ảnh) nhưng ép dùng eval_transform tạm thời để lấy nhãn phân chia
full_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=eval_transform)

# Tải tập Test (10.000 ảnh) - Dùng eval_transform
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=eval_transform)

# Lấy mảng nhãn (labels) để chia tỷ lệ đều (Stratified split)
train_labels = full_train_dataset.targets.numpy()
indices = np.arange(len(full_train_dataset))

# Chia 60.000 ảnh thành: 80% Train thực sự (48.000) và 20% Validation (12.000)
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=train_labels,
    random_state=42 # Cố định seed để chia giống nhau ở mọi lần chạy
)

print(f"Số lượng ảnh Train: {len(train_idx)}")
print(f"Số lượng ảnh Validation: {len(val_idx)}")
print(f"Số lượng ảnh Test (Giữ nguyên gốc): {len(test_dataset)}")

100.0%
100.0%
100.0%
100.0%

Số lượng ảnh Train: 48000
Số lượng ảnh Validation: 12000
Số lượng ảnh Test (Giữ nguyên gốc): 10000


## Gắn Transform chuẩn và Lưu phân chia xuống ổ cứng
Khởi tạo lại các tập Subset với đúng Transform của nó. Cuối cùng, lưu mảng `train_idx` và `val_idx` lại.

In [6]:
# Tải lại tập gốc nhưng áp dụng train_transform (có Augmentation)
raw_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=train_transform)

# Bọc các Subset
train_dataset = Subset(raw_train_dataset, train_idx)
val_dataset = Subset(full_train_dataset, val_idx) # full_train_dataset ở cell trước đã bọc eval_transform

# Lưu các index phân chia xuống ổ đĩa bằng PyTorch
torch.save(train_idx, 'data_splits/train_indices.pt')
torch.save(val_idx, 'data_splits/val_indices.pt')

# Chạy thử DataLoader xem hoạt động tốt không (Batch size nhỏ để test nhanh trên Mac)
temp_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
images, labels = next(iter(temp_loader))

print(f"Đã lưu các file phân chia vào thư mục 'data_splits/'.")
print(f"Kích thước 1 batch ảnh đầu ra: {images.shape}") # 1 batch 16 ảnh, 3 channel RGB, size 224x224

Đã lưu các file phân chia vào thư mục 'data_splits/'.
Kích thước 1 batch ảnh đầu ra: torch.Size([16, 3, 224, 224])
